# 02 — Build the county dataset

Reads the cached raw files, cleans them, merges the two ACS vintages on `GEOID`,
computes the affordability measures, attaches the geometry, and writes:

- `data/processed/counties.csv` — one row per county, no geometry
- `data/processed/counties.gpkg` — the same with geometry (EPSG:5070)
- `results/tables/headline.csv`, `extremes.csv`, `why_bins.csv`, `why_corr.csv`

**Price-to-income ratio** = median home value ÷ median household income: the number
of years of income it takes to buy the typical home. Its *change* between the two
windows is the variable the map shows.

In [15]:
from pathlib import Path
import json

import numpy as np
import pandas as pd
import geopandas as gpd

ROOT = Path.cwd().parent
RAW = ROOT / "data" / "raw"
PROCESSED = ROOT / "data" / "processed"
TABLES = ROOT / "results" / "tables"
for d in (PROCESSED, TABLES):
    d.mkdir(parents=True, exist_ok=True)

MIN_POP = 50_000      # analysis sample for the chart and the tables: US counties, no Puerto Rico
TOP_CODE = 2_000_001  # ACS top-code for median home value
NOT_CONUS = {"02", "15", "60", "66", "69", "72", "78"}  # AK, HI, PR and island areas: not on the map

## Clean each ACS vintage

In [16]:
def load_acs(year):
    df = pd.read_csv(RAW / f"acs5_{year}_county.csv", dtype=str)
    df["GEOID"] = df["state"].str.zfill(2) + df["county"].str.zfill(3)
    out = pd.DataFrame({"GEOID": df["GEOID"], "NAME": df["NAME"]})
    for col in df.columns:
        if col.startswith("B"):
            x = pd.to_numeric(df[col], errors="coerce")
            x = x.mask(x < 0)  # ACS uses large negative sentinels for suppressed estimates
            out[col] = x
    return out

acs19 = load_acs(2019).rename(columns={
    "B25077_001E": "home_value_2019", "B19013_001E": "income_2019", "B01003_001E": "pop_2019"})
acs24 = load_acs(2024).rename(columns={
    "B25077_001E": "home_value_2024", "B19013_001E": "income_2024", "B01003_001E": "pop_2024",
    "B08006_001E": "workers_2024", "B08006_017E": "wfh_2024"})

assert acs19["GEOID"].is_unique and acs24["GEOID"].is_unique
print(len(acs19), "counties in 2015-19;", len(acs24), "in 2020-24")
print("top-coded home values 2024:", int((acs24["home_value_2024"] >= TOP_CODE).sum()))

3220 counties in 2015-19; 3222 in 2020-24
top-coded home values 2024: 0


## Merge the two vintages

Connecticut replaced its counties with planning regions in 2022, so its 2020–24
GEOIDs have no 2015–19 match; a few Alaska boroughs changed too. Those rows are
counted here and left missing rather than patched.

In [17]:
merged = acs24.merge(acs19.drop(columns="NAME"), on="GEOID", how="outer", indicator=True)
print(merged["_merge"].value_counts())
unmatched = merged.loc[merged["_merge"] != "both", ["GEOID", "NAME", "_merge"]]
unmatched.to_csv(TABLES / "unmatched_geoids.csv", index=False)
df = merged.loc[merged["_merge"] == "both"].drop(columns="_merge").copy()

_merge
both          3211
left_only       11
right_only       9
Name: count, dtype: int64


## Affordability measures

- `ptir_*` — price-to-income ratio, in years of income
- `d_ptir` — change in the ratio between the windows (the map variable)
- `pct_*` — percentage changes in home value, income and population, as fractions
- `wfh_share_2024` — share of workers who worked from home, 2020–24

In [18]:
df["ptir_2019"] = df["home_value_2019"] / df["income_2019"]
df["ptir_2024"] = df["home_value_2024"] / df["income_2024"]
df["d_ptir"] = df["ptir_2024"] - df["ptir_2019"]
df["pct_home_value"] = df["home_value_2024"] / df["home_value_2019"] - 1
df["pct_income"] = df["income_2024"] / df["income_2019"] - 1
df["pct_pop"] = df["pop_2024"] / df["pop_2019"] - 1
df["wfh_share_2024"] = df["wfh_2024"] / df["workers_2024"]
df["STATEFP"] = df["GEOID"].str[:2]
df["in_sample"] = (df["pop_2024"] >= MIN_POP) & (df["STATEFP"] != "72")
df["conus"] = ~df["STATEFP"].isin(NOT_CONUS)

print(df[["ptir_2019", "ptir_2024", "d_ptir", "pct_pop"]].describe().round(2))
print("analysis sample:", int(df["in_sample"].sum()), "counties")

       ptir_2019  ptir_2024   d_ptir  pct_pop
count    3209.00    3205.00  3205.00  3211.00
mean        2.84       3.24     0.40     0.01
std         1.13       1.29     0.49     0.07
min         0.64       0.85    -2.64    -0.74
25%         2.15       2.42     0.15    -0.03
50%         2.57       2.95     0.37     0.01
75%         3.18       3.74     0.65     0.04
max        11.72      13.16     3.75     0.68
analysis sample: 990 counties


## Attach geometry and save

In [19]:
shp = next((RAW / "cb_2024_us_county_500k").glob("*.shp"))
counties = gpd.read_file(shp)[["GEOID", "STATEFP", "NAME", "STATE_NAME", "geometry"]]
print(counties.crs, len(counties), "polygons")

gdf = counties.drop(columns=["STATEFP", "NAME"]).merge(df, on="GEOID", how="left", validate="one_to_one")
# Polygons without data come out of the left merge with NaN flags, which turns the bool columns into
# object dtype; GeoPackage then stores them as the strings "True"/"False". Rebuild them as real bools.
gdf["STATEFP"] = gdf["GEOID"].str[:2]
gdf["conus"] = ~gdf["STATEFP"].isin(NOT_CONUS)  # from the polygon, so CT stays on the map (in grey)
gdf["in_sample"] = gdf["in_sample"].fillna(False).astype(bool)
gdf = gdf.to_crs("EPSG:5070")
print("polygons without data (CT, AK changes, PR):", int(gdf["d_ptir"].isna().sum()))

gdf.drop(columns="geometry").to_csv(PROCESSED / "counties.csv", index=False)
gdf.to_file(PROCESSED / "counties.gpkg", layer="counties", driver="GPKG")

EPSG:4269 3235 polygons
polygons without data (CT, AK changes, PR): 30


/var/folders/g5/9y1dddzs19d46jhcmn4bklrc0000gn/T/ipykernel_90086/3200066950.py:10: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  gdf["in_sample"] = gdf["in_sample"].fillna(False).astype(bool)


## Tables for the text

Every number quoted in the post comes from one of these files.

In [20]:
s = df.loc[df["in_sample"]]

headline = pd.DataFrame({
    "measure": [
        "median county price-to-income ratio, 2015-19",
        "median county price-to-income ratio, 2020-24",
        "median county change in price-to-income ratio (years of income)",
        "share of counties where the ratio rose",
        "median county change in home value (pct)",
        "median county change in household income (pct)",
        "median county change in population (pct)",
        "counties in analysis sample (pop >= 50,000, excl. Puerto Rico)",
    ],
    "value": [
        s["ptir_2019"].median(), s["ptir_2024"].median(), s["d_ptir"].median(),
        (s["d_ptir"] > 0).mean(),
        s["pct_home_value"].median() * 100, s["pct_income"].median() * 100,
        s["pct_pop"].median() * 100, len(s),
    ],
}).round(3)
headline.to_csv(TABLES / "headline.csv", index=False)

cols = ["NAME", "ptir_2019", "ptir_2024", "d_ptir", "pct_home_value", "pct_income", "pct_pop", "pop_2024"]
extremes = pd.concat([
    s.nlargest(10, "d_ptir")[cols].assign(group="largest increase"),
    s.nsmallest(10, "d_ptir")[cols].assign(group="largest decrease"),
]).round(3)
extremes.to_csv(TABLES / "extremes.csv", index=False)

# Binned means: 20 equal-count bins of population growth, mean change in the ratio within each
b = s[["pct_pop", "d_ptir", "pop_2024"]].dropna().copy()
b["bin"] = pd.qcut(b["pct_pop"], 20, labels=False, duplicates="drop")
why_bins = (b.groupby("bin")
              .agg(pct_pop=("pct_pop", "mean"), d_ptir=("d_ptir", "mean"), n=("d_ptir", "size"))
              .reset_index()).round(4)
why_bins.to_csv(TABLES / "why_bins.csv", index=False)

why_corr = pd.DataFrame({
    "pair": ["d_ptir vs pct_pop", "d_ptir vs ptir_2019 (catch-up < 0 < pull-away)", "d_ptir vs wfh_share_2024"],
    "corr": [s["d_ptir"].corr(s["pct_pop"]), s["d_ptir"].corr(s["ptir_2019"]), s["d_ptir"].corr(s["wfh_share_2024"])],
}).round(3)
why_corr.to_csv(TABLES / "why_corr.csv", index=False)

display(headline)
display(why_corr)
extremes

,measure,value
0,"median county price-to-income ratio, 2015-19",2.938
1,"median county price-to-income ratio, 2020-24",3.492
2,median county change in price-to-income ratio ...,0.527
3,share of counties where the ratio rose,0.977
4,median county change in home value (pct),48.444
5,median county change in household income (pct),26.694
6,median county change in population (pct),3.191
7,"counties in analysis sample (pop >= 50,000, ex...",990.000


,pair,corr
0,d_ptir vs pct_pop,0.455
1,d_ptir vs ptir_2019 (catch-up < 0 < pull-away),0.261
2,d_ptir vs wfh_share_2024,0.220


,NAME,ptir_2019,ptir_2024,d_ptir,pct_home_value,pct_income,pct_pop,pop_2024,group
1623,"Flathead County, Montana",4.870,7.244,2.374,0.957,0.316,0.108,110695.0,largest increase
570,"Bonner County, Idaho",5.058,7.284,2.226,0.919,0.333,0.171,51049.0,largest increase
560,"Kauai County, Hawaii",6.830,8.940,2.110,0.530,0.169,0.027,73731.0,largest increase
1995,"Watauga County, North Carolina",5.008,6.943,1.935,0.508,0.088,0.004,55123.0,largest increase
603,"Twin Falls County, Idaho",3.152,5.013,1.861,1.026,0.274,0.104,93734.0,largest increase
374,"Monroe County, Florida",7.055,8.897,1.842,0.580,0.253,0.080,81860.0,largest increase
589,"Kootenai County, Idaho",4.547,6.336,1.789,0.993,0.430,0.157,181996.0,largest increase
1624,"Gallatin County, Montana",5.378,7.138,1.760,0.870,0.409,0.148,124074.0,largest increase
575,"Canyon County, Idaho",3.420,5.099,1.679,1.187,0.467,0.152,250790.0,largest increase
571,"Bonneville County, Idaho",2.989,4.664,1.675,1.035,0.304,0.132,129523.0,largest increase
